# Fast GECToR と W&I による英語誤り検出・訂正

書籍 3-4「演習：文法誤り検出・訂正」に対応します。

このノートブックでは、[Fast GECToR](https://github.com/cofe-ai/fast-gector) という文法誤り検出・訂正システムを、英語学習者の作文に誤りの訂正を付けたコーパスで訓練し、英語の誤り訂正をしてみます。

データには、[BEA-2019 共有タスク](https://www.cl.cam.ac.uk/research/nl/bea2019st/)で公開された Write & Improve（W&I）コーパスと LOCNESS コーパス（W&I+LOCNESS v2.1）を使います。W&I は Cambridge English Write & Improve という英作文の学習サイトに投稿された作文に、誤りの訂正を付けたコーパスです。非営利の研究・教育目的なら、申請なしでダウンロードして使えます（ダウンロードした時点で、データに含まれるライセンスに同意したことになります）。

GPU を使うので、[ランタイム] → [ランタイムのタイプを変更] で [ハードウェア アクセラレータ] を `GPU`（T4 など）にしておいてください。

In [ ]:
import torch

if torch.cuda.is_available():
    print("使用可能な GPU:", torch.cuda.get_device_name(0))
else:
    print("警告: GPU が見つかりません（CPU で実行されます）。Colab の「ランタイム」→「ランタイムのタイプを変更」から GPU を選択してください。")

# インストール

Fast GECToR は GECToR の訓練を高速に行う派生版のシステムです。GitHub でソースコードが公開されているので、手元にダウンロードして使いましょう。

In [ ]:
%cd /content
![ -d fast-gector ] || git clone https://github.com/cofe-ai/fast-gector
%cd /content/fast-gector

Fast GECToR には、訓練の最初の数エポックだけエンコーダ（RoBERTa）を固定して出力層だけを訓練する機能（`--cold_step_count`）があります。ただし、固定を解除するときに DeepSpeed の設定ファイルから学習率を読み込む場所が間違っていて、エラーになってしまいます。そこで、ソースコードを1行だけ修正しておきます。

In [ ]:
!sed -i 's/self.lr = config\["optimizer"\].get("lr")/self.lr = config["optimizer"]["params"]["lr"]/' src/trainer.py
with open("src/trainer.py", encoding="utf-8") as f:
    trainer_src = f.read()
assert 'self.lr = config["optimizer"]["params"]["lr"]' in trainer_src, (
    "src/trainer.py のパッチ適用に失敗しました（上流コードが変更された可能性があります）"
)
!grep -n "self.lr = " src/trainer.py

Fast GECToR は2023年以降更新されておらず、少し古いバージョンの PyTorch（2.0）や DeepSpeed（0.10）、Transformers（4.31）を必要とします。2026年10月時点の Google Colab の Python は 3.13 で、これらのライブラリは Python 3.13 では動かないので、[uv](https://docs.astral.sh/uv/) という高速なパッケージ管理ツールを使って Python 3.10 の仮想環境を `/content/gector-env` に作り、そこにインストールします。数分かかります。最後に `True` と表示されれば GPU が使えます。

なお、以前のバージョンのノートブックでは NVIDIA の apex もインストールしていましたが、Fast GECToR では使われていないので、インストールしないことにしました。

In [ ]:
!pip install -q uv
!uv venv -q --clear -p 3.10 /content/gector-env
# DeepSpeed はインストール時に PyTorch を使うので、先に PyTorch を入れておく
!uv pip install -q -p /content/gector-env torch==2.0.1 numpy==1.25.1 "setuptools<70" wheel packaging ninja
!uv pip install -q -p /content/gector-env --no-build-isolation deepspeed==0.10.0
!uv pip install -q -p /content/gector-env -r requirements.txt
!/content/gector-env/bin/python -c "import sys, torch, deepspeed, transformers; print(sys.version, torch.__version__, deepspeed.__version__, transformers.__version__, torch.cuda.is_available())"

以下では `!deepspeed ...` のようにコマンドを実行するので、作った仮想環境のコマンドが使えるように PATH を設定しておきます。ランタイムを再起動した場合は、このセルをもう一度実行してください。

In [ ]:
%cd /content/fast-gector
import os
if "/content/gector-env/bin" not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = "/content/gector-env/bin:" + os.environ["PATH"]
!which deepspeed

# データの準備

W&I+LOCNESS v2.1 をダウンロードして展開します。ライセンスは `wi+locness/licence.wi.txt` と `wi+locness/license.locness.txt` に書かれています。

In [ ]:
!curl -fsSLO https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz
!tar xzf wi+locness_v2.1.bea19.tar.gz
!ls wi+locness/m2

誤りの訂正は M2 という形式で書かれています。`S` で始まる行が学習者の書いた文（単語分割済み）、`A` で始まる行がその文に対する訂正（何単語目から何単語目までを何に置き換えるか）です。

In [ ]:
!head -n 12 wi+locness/m2/ABC.train.gold.bea19.m2

Fast GECToR の訓練には「誤りを含む文」と「訂正後の文」のペアが必要なので、M2 の訂正を適用して訂正後の文を作ります。

W&I+LOCNESS の訓練データ（`ABC.train`、約3.4万文）を訓練データにします。テストデータの正解は公開されていないので、開発データ（`ABCN.dev`、約4,400文）をシャッフルして半分ずつに分け、開発データとテストデータにします。

In [ ]:
import random

def m2_to_parallel(path):
    """M2 ファイルを（誤りを含む文, 訂正後の文）のペアのリストに変換する"""
    pairs = []
    for block in open(path, encoding="utf-8").read().strip().split("\n\n"):
        lines = block.split("\n")
        src = lines[0][2:].split()  # 先頭の "S " を除く
        trg = list(src)
        offset = 0  # これまでの訂正で増減した単語数
        for line in lines[1:]:
            fields = line[2:].split("|||")  # 先頭の "A " を除く
            start, end = map(int, fields[0].split())
            error_type, correction, annotator = fields[1], fields[2], fields[5]
            if error_type in ("noop", "UNK") or annotator != "0":
                continue
            cor_tokens = correction.split()
            trg[start + offset:end + offset] = cor_tokens
            offset += len(cor_tokens) - (end - start)
        pairs.append((" ".join(src), " ".join(trg)))
    return pairs

train = m2_to_parallel("wi+locness/m2/ABC.train.gold.bea19.m2")
dev_test = m2_to_parallel("wi+locness/m2/ABCN.dev.gold.bea19.m2")
random.seed(0)
random.shuffle(dev_test)
half = len(dev_test) // 2
splits = {"train": train, "dev": dev_test[:half], "test": dev_test[half:]}

for name, pairs in splits.items():
    with open(f"{name}.src", "w", encoding="utf-8") as fs, open(f"{name}.trg", "w", encoding="utf-8") as ft:
        for src, trg in pairs:
            fs.write(src + "\n")
            ft.write(trg + "\n")
    print(name, len(pairs), "文（うち誤りを含む文", sum(src != trg for src, trg in pairs), "）")

In [ ]:
!paste train.src train.trg | head -n 5

GECToR は入力と出力のペアから編集の系列を抽出し、タグとして用います。Fast GECToR にはこの編集系列の抽出のためのスクリプトが含まれているので、こちらを使いましょう（上で作った仮想環境の Python で実行します）。数秒で終わります。

In [ ]:
!python utils/preprocess_data.py -s train.src -t train.trg -o train.edits
!python utils/preprocess_data.py -s dev.src -t dev.trg -o dev.edits
!wc -l train.edits dev.edits test.src test.trg

# モデルの訓練

作成したデータを用いて Fast GECToR モデルの訓練を行います。Fast GECToR は DeepSpeed というライブラリを使って訓練します。DeepSpeed の設定ファイルは `configs/` の下にありますが、Google Colab の GPU（T4）向けに以下のように変更した設定ファイルを用意します。

- `train_batch_size` は256のままで、`gradient_accumulation_steps` を8にして、一度に GPU に載せる文の数を32文にします（GPU のメモリが足りなくならないようにするため）。
- `torch_adam` を `true` にして、PyTorch の Adam を使います（DeepSpeed の Adam を使うと、Google Colab の CUDA のバージョンが合わずにコンパイルに失敗するため）。
- 学習率（`lr`）を元の設定の1e-5から2e-4に上げます。元の設定は、人工的に作った誤りデータで事前に訓練したモデルをさらに訓練するためのものです。この演習のように事前訓練なしで W&I だけで訓練する場合、1e-5 だとほとんど大文字・小文字の訂正しか覚えません。2026年10月に学習率を5e-5から5e-4まで変えて試したところ、2e-4 が一番良く、5e-4 にすると訓練がうまくいきませんでした。

In [ ]:
%%writefile configs/ds_config_colab.json
{
    "train_batch_size": 256,
    "gradient_accumulation_steps": 8,
    "optimizer": {
        "type": "Adam",
        "params": {
            "lr": 2e-4,
            "eps": 1e-8,
            "weight_decay": 1e-2,
            "torch_adam": true
        }
    },
    "gradient_clipping": 1.0,
    "fp16": {
        "enabled": true,
        "auto_cast": false,
        "loss_scale": 0,
        "initial_scale_power": 16,
        "loss_scale_window": 1000,
        "hysteresis": 2,
        "consecutive_hysteresis": false,
        "min_loss_scale": 1
    },
    "zero_optimization": {
        "stage": 1,
        "reduce_bucket_size": 5e8,
        "allgather_bucket_size": 5e8
    }
}

訓練には `train.py` を用います（`scripts/train.sh` の中身とほぼ同じですが、Google Colab 向けにいくつかオプションを変えています）。

- `--num_epochs` は訓練のエポック数です。ここでは10エポックにしています。2026年10月に試したときは、T4 で40分ほどかかりました（最初の2エポックはエンコーダを固定するので速く進みます）。
- `--cold_step_count` は、最初の何エポックの間エンコーダを固定して、新しく追加した出力層だけを `--cold_lr` の学習率で訓練するかを指定します。出力層はランダムな値から始まるので、最初からエンコーダも一緒に訓練するより、出力層をある程度訓練してからエンコーダも訓練するほうが良い結果になります。ここでは GECToR の訓練ガイドと同じく2エポックにしています。
- `--eval_interval` ステップごとに開発データで評価し、`--save_interval` ステップごとにモデルを `ckpts/` の下に保存します。両方を同じ値にしておくと、開発データで一番良かったモデルが必ず保存されます。
- `--tn_prob` は「訂正の必要がない文を訓練に使う確率」をコントロールします。これを変化させることで precision/recall のバランスを取ることができます。いくつかのステージに分けて訓練し、最初の訓練時には `--tn_prob 0` として、最後に適切な値に調整すると良いようです。また、`--skip_correct` もデフォルトは0ですが1にした方が良いようです。詳しくは [GECToR の訓練ガイド](https://github.com/grammarly/gector/blob/master/docs/training_parameters.md)をご覧ください。

訓練の様子は TensorBoard で見ることができます。

### 実行モードの切り替え（`QUICK_RUN`）

まずは短時間で動作確認したい場合は、下のセルで `QUICK_RUN = True` に変更してください。訓練データを 5,000 文・3 エポックに絞って実行するため、T4 でも 5〜10 分ほどで訓練が終わります。本格的な評価を行う場合は `QUICK_RUN = False`（W&I 全 3.4 万文・10 エポック、T4 で約 40 分）のまま実行してください。

In [ ]:
import os

# True: 5,000 文・3 エポックの軽量モード（T4 で約 5〜10 分）
# False: W&I 全 3.4 万文・10 エポックの通常モード（T4 で約 40 分）
QUICK_RUN = False

if QUICK_RUN:
    with open("train.edits", encoding="utf-8") as fin, open("train_quick.edits", "w", encoding="utf-8") as fout:
        for i, line in enumerate(fin):
            if i >= 5000:
                break
            fout.write(line)
    os.environ["TRAIN_FILE"] = "train_quick.edits"
    os.environ["NUM_EPOCHS"] = "3"
    os.environ["COLD_STEP_COUNT"] = "1"
    # 軽量モードでも複数のチェックポイントを比較できる間隔にする。
    os.environ["EVAL_INTERVAL"] = "15"
else:
    os.environ["TRAIN_FILE"] = "train.edits"
    os.environ["NUM_EPOCHS"] = "10"
    os.environ["COLD_STEP_COUNT"] = "2"
    os.environ["EVAL_INTERVAL"] = "100"

print(f"QUICK_RUN={QUICK_RUN} | TRAIN_FILE={os.environ['TRAIN_FILE']} | NUM_EPOCHS={os.environ['NUM_EPOCHS']}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
%%bash
set -o pipefail
timestamp=`date "+%Y%m%d_%H%M%S"`
save_dir="ckpts/ckpt_$timestamp"
mkdir -p $save_dir
deepspeed --master_port 49828 train.py \
    --deepspeed \
    --deepspeed_config configs/ds_config_colab.json \
    --num_epochs "${NUM_EPOCHS:-10}" \
    --max_num_tokens 128 \
    --valid_batch_size 64 \
    --cold_step_count "${COLD_STEP_COUNT:-2}" \
    --warmup 0.1 \
    --cold_lr 1e-3 \
    --skip_correct 0 \
    --skip_complex 0 \
    --sub_token_mode average \
    --special_tokens_fix 1 \
    --unk2keep 0 \
    --tp_prob 1 \
    --tn_prob 1 \
    --detect_vocab_path ./data/vocabulary/d_tags.txt \
    --correct_vocab_path ./data/vocabulary/labels.txt \
    --do_eval \
    --train_path "${TRAIN_FILE:-train.edits}" \
    --valid_path dev.edits \
    --save_dir $save_dir \
    --use_cache 0 \
    --log_interval 10 \
    --eval_interval "${EVAL_INTERVAL:-100}" \
    --save_interval "${EVAL_INTERVAL:-100}" \
    --pretrained_transformer_path roberta-base \
    --tensorboard_dir logs/tb/gector_$timestamp \
    2>&1 | tee $save_dir/train.log | grep --line-buffered -v "it/s"

# モデルの評価

訓練したモデルを評価しましょう。訓練されたモデルは `ckpts/ckpt_時刻/globalstep-XXX` のような名前で保存されています。開発データでの学習曲線（TensorBoard）を見るとどのモデルが適しているかがわかります。ここでは、訓練中に記録された評価結果から、開発データでの正解率が一番高かったモデルを選びます。

In [ ]:
import glob, json, os

valid_dirs = [d for d in sorted(glob.glob("ckpts/ckpt_*")) if glob.glob(f"{d}/metrics_globalstep-*.json")]
assert valid_dirs, "metrics_globalstep-*.json が見つかりません"
save_dir = valid_dirs[-1]
metrics_files = sorted(glob.glob(f"{save_dir}/metrics_globalstep-*.json"), key=lambda p: int(p.split("-")[-1].split(".")[0]))
best = json.load(open(metrics_files[-1]))
ckpt_path = f"{save_dir}/globalstep-{best['best_global_step']}"
print(best)
print(ckpt_path, os.path.exists(ckpt_path))
assert os.path.exists(ckpt_path), f"チェックポイントが見つかりません: {ckpt_path}"
os.environ["CKPT_PATH"] = ckpt_path

`predict.py` でテストデータの誤りを訂正します（`scripts/predict.sh` の中身とほぼ同じです）。

`--min_error_probability` は、単語ごとの誤り検出スコアと編集候補のスコアに適用する閾値です。閾値を上げると通常、適合率が上がり、再現率が下がります。`--additional_confidence` は「訂正しない」というタグの確率に足す値で、これも大きくすると訂正が控えめになります。2026年10月に開発データで試したところ、`--min_error_probability 0.3`、`--additional_confidence 0` が一番良かったので、ここではその値を使います。

In [ ]:
%%bash
set -euo pipefail
deepspeed --include localhost:0 --master_port 42991 predict.py \
    --batch_size 64 \
    --iteration_count 5 \
    --min_seq_len 3 \
    --max_num_tokens 128 \
    --min_error_probability 0.3 \
    --additional_confidence 0.0 \
    --sub_token_mode average \
    --max_pieces_per_token 5 \
    --ckpt_path "$CKPT_PATH" \
    --deepspeed_config configs/ds_config_colab.json \
    --detect_vocab_path ./data/vocabulary/d_tags.txt \
    --correct_vocab_path ./data/vocabulary/labels.txt \
    --pretrained_transformer_path roberta-base \
    --input_path test.src \
    --out_path test.pred \
    --special_tokens_fix 1 \
    --detokenize 0 \
    --segmented 1 \
    2>&1 | tail -n 5

入力（`test.src`）とシステムの出力（`test.pred`）の差分を見てみましょう。`-` で始まる行が入力、`+` で始まる行がシステムが訂正した文です。

In [ ]:
!diff -u test.src test.pred | head -n 200

### 訂正結果の誤り分析テーブル

`diff` だけでは正解（`test.trg`）との違いが追いにくいため、原文（`src`）・システム予測（`pred`）・正解（`trg`）を並べた表を作成し、各文を「完全一致訂正」「部分・過剰訂正」「見逃し」「正解通り無訂正」に分類して確認します。

In [ ]:
import pandas as pd

with open("test.src", encoding="utf-8") as fs, open("test.pred", encoding="utf-8") as fp, open("test.trg", encoding="utf-8") as ft:
    srcs = [l.rstrip("\n") for l in fs]
    preds = [l.rstrip("\n") for l in fp]
    trgs = [l.rstrip("\n") for l in ft]

def classify_edit(src, pred, trg):
    if src == trg and pred == trg:
        return "正解通り無訂正"
    if src != trg and pred == trg:
        return "完全一致訂正"
    if src != trg and pred == src:
        return "見逃し"
    return "部分・過剰訂正"

df_err = pd.DataFrame({
    "原文 (src)": srcs,
    "システム予測 (pred)": preds,
    "正解 (trg)": trgs,
    "分類": [classify_edit(s, p, t) for s, p, t in zip(srcs, preds, trgs)],
})
print(df_err["分類"].value_counts())

for cat in ("完全一致訂正", "部分・過剰訂正", "見逃し"):
    print(f"\n=== {cat}（先頭 5 件）===")
    display(df_err[df_err["分類"] == cat].head(5))

## ERRANT による評価

文法誤り訂正では、[ERRANT](https://github.com/chrisjbryant/errant) というツールで、システムの訂正と正解（参照訳）の訂正を比べて precision、recall、F0.5 を計算するのが一般的です。ERRANT をインストールして評価してみましょう。

2026年10月に T4 で試したときは、precision が0.51、recall が0.24、F0.5 が0.41でした（実行のたびに結果は少し変わり、A100 で試したときは F0.5 が0.44でした）。最初のバージョンのノートブック（学習率5e-5、5エポック、`--cold_step_count 0`）では F0.5 が0.26～0.28程度だったので、学習率・エポック数・`--cold_step_count`・`--min_error_probability` を調整することで大きく改善しています。さらに、cLang-8 のような大きなデータで先に訓練してから W&I で訓練すると、もっと良くなります。

In [ ]:
# ERRANT は Google Colab の Python にインストールする（PATH の先頭にある仮想環境の Python ではなく）
import sys
!{sys.executable} -m pip install -q errant
!{sys.executable} -m spacy download en_core_web_sm
!errant_parallel -orig test.src -cor test.pred -out hyp.m2
!errant_parallel -orig test.src -cor test.trg -out ref.m2
!errant_compare -hyp hyp.m2 -ref ref.m2

## 閾値（`--min_error_probability`）による Precision / Recall トレードオフ

文法誤り訂正では、正しい文を誤って書き換えてしまう（偽陽性）よりも見逃し（偽陰性）のほうが害が小さいため、`ch1_evaluation.ipynb` でも扱ったように 再現率より適合率を重視する $F_{0.5}$ が評価尺度として使われます。

軽量モードでは開発データの先頭 500 文、通常モードでは開発データ全体に対して `--min_error_probability` を `0.1, 0.3, 0.5, 0.7` と変えて推論し、ERRANT で Precision・Recall・$F_{0.5}$ がどのように変化するかを確認しましょう。

従来のチェックポイント（開発データで正解率が最高）に加え、同じ訓練で保存した他のチェックポイントから中ほどと最後のものも候補にします。すべての候補を同じ開発データの文で評価し、ERRANT の $F_{0.5}$ が最も高いチェックポイントと閾値の組を選びます。同点の場合は従来のチェックポイントを優先し、その次に従来の閾値 `0.3` に近いものを選びます。評価に使う文数は `N_DEV` で変更できます。

In [ ]:
from itertools import islice
from pathlib import Path
import re
import subprocess
from tempfile import TemporaryDirectory

import pandas as pd


def run_prediction(checkpoint, threshold, source_path, pred_path):
    subprocess.run([
        "deepspeed", "--include", "localhost:0", "--master_port", "42992", "predict.py",
        "--batch_size", "64",
        "--iteration_count", "5",
        "--min_seq_len", "3",
        "--max_num_tokens", "128",
        "--min_error_probability", str(threshold),
        "--additional_confidence", "0.0",
        "--sub_token_mode", "average",
        "--max_pieces_per_token", "5",
        "--ckpt_path", str(checkpoint),
        "--deepspeed_config", "configs/ds_config_colab.json",
        "--detect_vocab_path", "./data/vocabulary/d_tags.txt",
        "--correct_vocab_path", "./data/vocabulary/labels.txt",
        "--pretrained_transformer_path", "roberta-base",
        "--input_path", str(source_path),
        "--out_path", str(pred_path),
        "--special_tokens_fix", "1",
        "--detokenize", "0",
        "--segmented", "1",
    ], check=True, stdout=subprocess.DEVNULL)


def annotate_edits(source_path, corrected_path, m2_path):
    subprocess.run([
        "errant_parallel", "-orig", str(source_path), "-cor", str(corrected_path),
        "-out", str(m2_path),
    ], check=True)


def errant_scores(hyp_path, ref_path):
    result = subprocess.run(
        ["errant_compare", "-hyp", str(hyp_path), "-ref", str(ref_path)],
        check=True, capture_output=True, text=True,
    )
    match = re.search(
        r"^\s*(\d+)\s+(\d+)\s+(\d+)\s+[0-9.]+\s+[0-9.]+\s+[0-9.]+\s*$",
        result.stdout, re.MULTILINE,
    )
    if match is None:
        raise ValueError(f"errant_compare の出力を解析できませんでした:\n{result.stdout}")
    tp, fp, fn = map(int, match.groups())
    # ERRANT の表示で丸められる前の F0.5 を、編集の件数から求める。
    return {
        "TP": tp, "FP": fp, "FN": fn,
        "Precision": tp / (tp + fp) if tp + fp else 0.0,
        "Recall": tp / (tp + fn) if tp + fn else 0.0,
        "F0.5": 5 * tp / (5 * tp + 4 * fp + fn) if tp + fp + fn else 0.0,
    }


def evaluate_prediction(checkpoint, threshold, source_path, ref_path, output_dir):
    pred_path = output_dir / "pred.txt"
    hyp_path = output_dir / "hyp.m2"
    run_prediction(checkpoint, threshold, source_path, pred_path)
    annotate_edits(source_path, pred_path, hyp_path)
    return errant_scores(hyp_path, ref_path)


baseline_checkpoint = Path(ckpt_path)
baseline_threshold = 0.3
other_checkpoints = sorted(
    (
        path for path in baseline_checkpoint.parent.glob("globalstep-*")
        if path.is_dir() and path.name.removeprefix("globalstep-").isdigit()
        and path != baseline_checkpoint
    ),
    key=lambda path: int(path.name.removeprefix("globalstep-")),
)
candidates = [baseline_checkpoint]
if other_checkpoints:
    middle = other_checkpoints[(len(other_checkpoints) - 1) // 2]
    candidates.extend(dict.fromkeys((middle, other_checkpoints[-1])))

# 評価する文と参照の M2 ファイルは、すべての候補で共通。
N_DEV = 500 if QUICK_RUN else None
dev_results = []
with TemporaryDirectory(prefix="gector-dev-") as directory:
    output_dir = Path(directory)
    dev_src, dev_trg = output_dir / "dev.src", output_dir / "dev.trg"
    for target in (dev_src, dev_trg):
        with open(target.name, encoding="utf-8") as lines:
            target.write_text("".join(islice(lines, N_DEV)), encoding="utf-8")
    dev_ref = output_dir / "ref.m2"
    annotate_edits(dev_src, dev_trg, dev_ref)
    for checkpoint in candidates:
        for threshold in (0.1, baseline_threshold, 0.5, 0.7):
            scores = evaluate_prediction(checkpoint, threshold, dev_src, dev_ref, output_dir)
            dev_results.append({
                "checkpoint": checkpoint.name,
                "min_error_probability": threshold,
                **scores,
            })

selected = max(dev_results, key=lambda row: (
    row["F0.5"],
    row["checkpoint"] == baseline_checkpoint.name,
    -abs(row["min_error_probability"] - baseline_threshold),
))
display(pd.DataFrame(dev_results).round(4))
print("開発データで選んだ設定:", selected["checkpoint"], selected["min_error_probability"])

## 同じテストデータで従来設定と比較

上で選んだ組を固定し、従来設定（開発データの正解率が最高のチェックポイント、閾値 `0.3`）と、同じ `test.src` と参照 `ref.m2` で比較します。開発データでの改善がテストデータでも得られるか、適合率・再現率・$F_{0.5}$ を見比べてみましょう。

In [ ]:
baseline_test = errant_scores("hyp.m2", "ref.m2")
selected_checkpoint = baseline_checkpoint.parent / selected["checkpoint"]
selected_threshold = selected["min_error_probability"]

if (selected_checkpoint, selected_threshold) == (baseline_checkpoint, baseline_threshold):
    selected_test = baseline_test
else:
    with TemporaryDirectory(prefix="gector-test-") as directory:
        selected_test = evaluate_prediction(
            selected_checkpoint, selected_threshold, Path("test.src"), Path("ref.m2"),
            Path(directory),
        )

pd.DataFrame([
    {
        "設定": "従来（開発正解率・閾値 0.3）",
        "checkpoint": baseline_checkpoint.name,
        "min_error_probability": baseline_threshold,
        **baseline_test,
    },
    {
        "設定": "開発 ERRANT F0.5 で選択",
        "checkpoint": selected_checkpoint.name,
        "min_error_probability": selected_threshold,
        **selected_test,
    },
]).round(4)